# 05 — SHAP Interpretability

Compute SHAP values for XGBoost and logistic regression models.

In [1]:
import sys
sys.path.insert(0, '..')
import numpy as np
from sklearn.model_selection import train_test_split
from src.config import *
from src.data_extraction import load_all_data
from src.preprocessing import preprocess_expression_matrix, prepare_ml_arrays
from src.models import get_model_definitions, get_endosomal_features
from src.interpretability import compute_shap_values, shap_feature_importance, save_shap_values
from src.figures import set_publication_style, plot_shap_beeswarm, plot_shap_bar
set_publication_style()

In [2]:
data = load_all_data()
matrix = preprocess_expression_matrix(data['mouse_matrix'])
X, y, names = prepare_ml_arrays(matrix)
endo = get_endosomal_features(names)
endo_idx = [names.index(f) for f in endo if f in names]
X_endo = X[:, endo_idx]
print(f'Endosomal features: {len(endo)}')
print(endo[:10])

[data_extraction] Loading real data from aba6334_data_file_s1.xlsx and aba6334_data_file_s3.xlsx


/Users/mohamedsmacbookair/Programs/Research/edosomal-biomarker-ml/notebooks/../src/data_extraction.py:94: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  data = data.replace({"undetected": np.nan, "NaN": np.nan, "nan": np.nan})
/Users/mohamedsmacbookair/Programs/Research/edosomal-biomarker-ml/notebooks/../src/data_extraction.py:94: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  data = data.replace({"undetected": np.nan, "NaN": np.nan, "nan": np.nan})
/Users/mohamedsmacbookair/Programs/Research/edosomal-biomarker-ml/notebooks/../src/d

  Proteins:      1505
  Significant:   71
  Mouse samples: 11
  Human subjects:58
  Data source:   supplement
Endosomal features: 27
['Cadm4', 'Aplp2', 'App', 'Aplp1', 'Chl1', 'Bsg', 'Cadm1', 'Clstn1', 'Clstn3', 'Cntn1']


In [3]:
X_tr, X_te, y_tr, y_te = train_test_split(
    X_endo, y, test_size=0.3, random_state=42, stratify=y)
print(f'Train: {X_tr.shape}, Test: {X_te.shape}')

Train: (7, 27), Test: (4, 27)


In [4]:
# Fit XGBoost
models = get_model_definitions()
xgb_pipe = models['Endosomal-XGB']
xgb_pipe.fit(X_tr, y_tr)
print('XGBoost trained')

XGBoost trained


In [5]:
# SHAP values
shap_vals, expected_val = compute_shap_values(xgb_pipe, X_tr, X_te, endo)
print(f'SHAP values shape: {shap_vals.shape}')
importance = shap_feature_importance(shap_vals, endo)
print('\nTop 10 features:')
print(importance.head(10).to_string(index=False))

SHAP values shape: (4, 27)

Top 10 features:
feature  mean_abs_shap  rank
  Cadm4            0.0     1
   Mapt            0.0     2
  Sort1            0.0     3
  Sorl1            0.0     4
  Sez6l            0.0     5
   Sez6            0.0     6
  Opcml            0.0     7
    Ntm            0.0     8
  Nrxn1            0.0     9
  Nrcam            0.0    10


In [6]:
# Save
save_shap_values(shap_vals, endo, 'Endosomal-XGB')
importance.to_csv(RESULTS_DIR / 'shap_importance_xgb.csv', index=False)

[interpretability] Saved SHAP values to /Users/mohamedsmacbookair/Programs/Research/edosomal-biomarker-ml/notebooks/../results/shap_values_Endosomal-XGB.csv


In [7]:
# Bar chart
plot_shap_bar(importance)
print('SHAP bar chart saved')

[figures] Saved fig3b_shap_bar
SHAP bar chart saved


In [8]:
# Beeswarm (requires shap installed)
try:
    plot_shap_beeswarm(shap_vals, X_te, endo)
    print('SHAP beeswarm saved')
except Exception as e:
    print(f'Beeswarm skipped: {e}')

[figures] Saved fig3a_shap_beeswarm
SHAP beeswarm saved


In [9]:
# Logistic regression SHAP
lr_pipe = models['Endosomal-Optimized']
lr_pipe.fit(X_tr, y_tr)
shap_lr, ev_lr = compute_shap_values(lr_pipe, X_tr, X_te, endo, model_type='linear')
imp_lr = shap_feature_importance(shap_lr, endo)
print('\nTop 10 features (LR SHAP):')
print(imp_lr.head(10).to_string(index=False))

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/shap/explainers/_linear.py:99: FutureWarning: The feature_perturbation option is now deprecated in favor of using the appropriate masker (maskers.Independent, maskers.Partition or maskers.Impute).
  warnings.warn(wmsg, FutureWarning)


Estimating transforms:   0%|          | 0/1000 [00:00<?, ?it/s]


Top 10 features (LR SHAP):
feature  mean_abs_shap  rank
  Lamp2       1.005699     1
   Ctsb       0.766153     2
  Ncam1       0.675168     3
  Nrxn1       0.623602     4
  Aplp2       0.568452     5
  Sort1       0.538426     6
  Sez6l       0.444471     7
  Cadm1       0.437886     8
    App       0.435150     9
  Cntn1       0.418282    10
